# 08 · Compare one-hot and compact policy encodings

## Question

How much does the choice of encoding change the quantum resources required to represent the same policy-synthesis problem?

## Concept

Let $N_\rho$ count regimes and $N_\psi$ count policies. The one-hot penalties follow the QUBO construction in [1].

**Informatics view.** An encoding assigns a binary string to each policy map. One-hot uses one bit $q_{ij}$ per regime–policy pair. The selected policy is the single 1 in each row, so $\sum_j q_{ij}=1$. The local score is linear in these bits. A quadratic penalty enforces the row constraint and cross-policy bit pairs encode the switching penalty. One-hot uses $N_\rho N_\psi$ bits.

Compact encoding stores each policy index in $b=\lceil\log_2N_\psi\rceil$ bits, using $N_\rho b$ in total. With three policies, 00, 01 and 10 are valid and 11 is invalid. We penalize invalid codes locally. Both encodings represent the same $N_\psi^{N_\rho}$ valid maps.

**Physics / quantum view.** Each bit becomes a qubit. Under $x=(1-Z)/2$, a product involving three or more bits can generate a Pauli term acting on three or more qubits. We call this a higher-order interaction. These products may be implemented directly as multi-qubit parity rotations. Alternatively, quadratization replaces the higher-order binary objective with a quadratic one by adding auxiliary variables, or ancillas, and terms that enforce their intended values. The auxiliary bits can reduce the qubit savings.

## Worked example

Use the same chain, mismatch cost and switching weight $\beta=2$ as Notebooks 06–07. Maximize the same local score minus $\beta$ for each switching edge. The 4×2 and 4×3 cases use the original illustrative matrices. Larger cases extend the table deterministically. Each pair of encodings uses the identical instance.

For two policies, each regime needs one compact bit $x_i$. The local energy is $-J_{i0}+(J_{i0}-J_{i1})x_i$ and the switching cost is $\beta(x_i+x_k-2x_ix_k)$. This is a clean QUBO.

For two or more bits, the equality test on an edge is the product of equalities for each bit position. For two bits per regime, the mismatch is
$$1-(1-x_{i0}-x_{k0}+2x_{i0}x_{k0})(1-x_{i1}-x_{k1}+2x_{i1}x_{k1}).$$
The variables $x_{i0},x_{i1}$ are the two bits in regime $i$, and $k$ is the neighboring regime. The equality test is dimensionless. Multiplying the mismatch by $\beta$ expresses it in the same score units as $J$.

It contains cubic and quartic terms, including a quartic coefficient of $-4\beta$. Three bits per regime can generate degree six. For three policies, the additional invalid-code penalty is $A x_{i0}x_{i1}$. The edge formula is used even on invalid strings as an extension of the valid-map objective. The penalty keeps these states above the feasible optimum.

## Exercise / prediction

How many 4×3 compact strings are invalid? Predict whether the 8×4 compact encoding uses fewer qubits and fewer entangling gates under the same parity-rotation implementation.

In [1]:
from itertools import combinations, product
from math import comb

import numpy as np
from IPython.display import Markdown, display

phase7 = np.array([[4, 0, 2], [1, 4, 3], [4, 0, 2],
                   [0, 6, 3], [5, 1, 2], [1, 4, 2]])
scores = np.fromfunction(lambda i, j: (3*i + 2*j + i*j) % 7, (8, 8), dtype=int)
scores[:6, :3] = phase7
beta = 2
cases = [(4, 2), (4, 3), (4, 4), (8, 4), (8, 8)]

def add_truth_table(polynomial, variables, values):
    # Coefficient for product S: alternating sum of truth-table values on subsets of S.
    for mask in range(len(values)):
        term = tuple(i for b, i in enumerate(variables) if mask & (1 << b))
        coefficient = 0
        subset = mask
        while True:
            coefficient += (-1)**((mask ^ subset).bit_count()) * values[subset]
            if subset == 0:
                break
            subset = (subset - 1) & mask
        polynomial[term] = polynomial.get(term, 0) + coefficient

def build_encodings(J):
    n, m = J.shape
    b = (m - 1).bit_length()
    A = int(J.sum()) + 1
    onehot = {(): n*A}
    for i in range(n):
        for j in range(m):
            onehot[(i*m + j,)] = -int(J[i, j]) - A
        for j, k in combinations(range(m), 2):
            onehot[(i*m + j, i*m + k)] = 2*A
    for i in range(n - 1):
        for j, k in product(range(m), repeat=2):
            if j != k:
                onehot[(i*m + j, (i + 1)*m + k)] = beta

    compact = {}
    codes = 2**b
    for i in range(n):
        variables = range(i*b, (i + 1)*b)
        local = [-int(value) for value in J[i]] + [A] * (codes - m)
        add_truth_table(compact, variables, local)
    for i in range(n - 1):
        variables = range(i*b, (i + 2)*b)
        mismatch = [beta * (s % codes != s // codes) for s in range(codes**2)]
        add_truth_table(compact, variables, mismatch)
    return (m, onehot), (b, {term: weight for term, weight in compact.items() if weight})

def pauli_coefficients(polynomial):
    # Substitute x_i=(1-Z_i)/2 into each product and combine identical Pauli strings.
    pauli = {}
    for term, weight in polynomial.items():
        for order in range(len(term) + 1):
            for subset in combinations(term, order):
                coefficient = weight * (-1)**order / 2**len(term)
                pauli[subset] = pauli.get(subset, 0) + coefficient
    return {term: weight for term, weight in pauli.items() if weight}

## Exact verification on small cases

The two encodings use different invalid strings, so compare decoded valid maps and their energies. For each encoding, verify that all ground states are feasible. With nonnegative scores, a constant valid map has energy at most zero. The penalty $A=1+\sum J_{ij}$ makes every invalid string cost strictly more than zero even after local rewards, since the switching terms are nonnegative in the original objectives.

Both binary and Pauli energies are evaluated for every string in the three 4-regime cases. For the 8-regime cases, only coefficients are constructed and counted.

In [2]:
def verify(J, width, polynomial, pauli, encoding):
    n, m = J.shape
    qubits = n*width
    bits = (np.arange(2**qubits)[:, None] >> np.arange(qubits)) & 1
    rows = bits.reshape(-1, n, width)
    if encoding == "One-hot":
        feasible = (rows.sum(axis=2) == 1).all(axis=1)
        decoded = rows.argmax(axis=2)
    else:
        decoded = rows @ (1 << np.arange(width))
        feasible = (decoded < m).all(axis=1)

    energies = np.zeros(len(bits))
    for term, weight in polynomial.items():
        energies += weight * bits[:, term].prod(axis=1)
    spins = 1 - 2*bits
    ising = np.zeros(len(bits))
    for term, weight in pauli.items():
        ising += weight * spins[:, term].prod(axis=1)
    np.testing.assert_array_equal(energies, ising)

    expected = {p: int(sum(J[i, j] for i, j in enumerate(p))) -
                beta*sum(a != b for a, b in zip(p, p[1:]))
                for p in product(range(m), repeat=n)}
    actual = {tuple(p): -energy for p, energy in zip(decoded[feasible], energies[feasible], strict=True)}
    assert feasible.sum() == m**n and actual == expected
    best = max(expected.values())
    ground = energies == energies.min()
    assert np.all(feasible[ground]) and energies.min() == -best
    assert {tuple(p) for p in decoded[ground]} == {p for p, value in expected.items() if value == best}
    winners = [p for p, value in expected.items() if value == best]
    return best, winners

results = []
for n, m in cases:
    J = scores[:n, :m]
    optima = []
    for label, (width, polynomial) in zip(("One-hot", "Compact"), build_encodings(J), strict=True):
        pauli = pauli_coefficients(polynomial)
        optimum = verify(J, width, polynomial, pauli, label) if n == 4 else None
        optima.append(optimum)
        results.append((n, m, label, n*width, polynomial, pauli, optimum))
    assert optima[0] == optima[1]

## Result · representation and cost-layer resources

The table counts nonzero terms after collecting like terms. The constant is excluded. $ZZ$ counts weight-two Pauli terms even when the full Hamiltonian has higher-order terms. The last resource column estimates a cost layer using one RZ rotation per nonconstant Pauli term and a parity ladder of $2(k-1)$ CNOT gates for a weight-$k$ term. Counts assume independent term synthesis with all-to-all logical connectivity, before gate cancellation, circuit scheduling, or routing. They are gate counts for one cost layer and exclude the mixer and measurement.

In [3]:
lines = ["| Size | Encoding | Qubits | Binary strings | Feasible | Fraction | Binary linear | Binary quad. | Binary higher | ZZ | Pauli weight ≥3 | Cost RZ / CNOT | Exact result |",
         "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|"]
for n, m, label, qubits, polynomial, pauli, optimum in results:
    states, valid = 2**qubits, m**n
    linear = sum(len(term) == 1 for term in polynomial)
    quadratic = sum(len(term) == 2 for term in polynomial)
    higher = sum(len(term) >= 3 for term in polynomial)
    zz = sum(len(term) == 2 for term in pauli)
    higher_pauli = sum(len(term) >= 3 for term in pauli)
    rz = sum(bool(term) for term in pauli)
    cnot = sum(2*(len(term) - 1) for term in pauli if len(term) >= 2)
    if optimum is None:
        exact = "Analytical counts only"
    else:
        best, winners = optimum
        maps = ", ".join("".join(str(int(j) + 1) for j in p) for p in winners)
        exact = f"Agrees: {maps}, score {best}"
    lines.append(f"| {n}×{m} | {label} | {qubits} | {states:,} | {valid:,} | {valid/states:.3g} | "
                 f"{linear} | {quadratic} | {higher} | {zz} | {higher_pauli} | {rz} / {cnot} | {exact} |")
display(Markdown("\n".join(lines)))

| Size | Encoding | Qubits | Binary strings | Feasible | Fraction | Binary linear | Binary quad. | Binary higher | ZZ | Pauli weight ≥3 | Cost RZ / CNOT | Exact result |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|
| 4×2 | One-hot | 8 | 256 | 16 | 0.0625 | 8 | 10 | 0 | 10 | 0 | 18 / 20 | Agrees: 1112, score 13 |
| 4×2 | Compact | 4 | 16 | 16 | 1 | 4 | 3 | 0 | 3 | 0 | 7 / 6 | Agrees: 1112, score 13 |
| 4×3 | One-hot | 12 | 4,096 | 81 | 0.0198 | 12 | 30 | 0 | 30 | 0 | 42 / 60 | Agrees: 1112, score 13 |
| 4×3 | Compact | 8 | 256 | 81 | 0.316 | 8 | 16 | 15 | 10 | 3 | 21 / 38 | Agrees: 1112, score 13 |
| 4×4 | One-hot | 16 | 65,536 | 256 | 0.00391 | 16 | 60 | 0 | 60 | 0 | 76 / 120 | Agrees: 4442, score 19 |
| 4×4 | Compact | 8 | 256 | 256 | 1 | 8 | 16 | 15 | 10 | 3 | 19 / 38 | Agrees: 4442, score 19 |
| 8×4 | One-hot | 32 | 4,294,967,296 | 65,536 | 1.53e-05 | 32 | 132 | 0 | 132 | 0 | 164 / 264 | Analytical counts only |
| 8×4 | Compact | 16 | 65,536 | 65,536 | 1 | 15 | 35 | 35 | 21 | 7 | 42 / 84 | Analytical counts only |
| 8×8 | One-hot | 64 | 18,446,744,073,709,551,616 | 16,777,216 | 9.09e-13 | 64 | 616 | 0 | 616 | 0 | 680 / 1232 | Analytical counts only |
| 8×8 | Compact | 24 | 16,777,216 | 16,777,216 | 1 | 23 | 86 | 287 | 44 | 36 | 104 / 316 | Analytical counts only |

**Exercise check.** The 4×3 compact encoding has 256 strings and 81 feasible maps, leaving 175 invalid strings. In the 8×4 and 8×8 examples, compact uses fewer qubits and fewer estimated CNOTs overall. Its higher-weight terms individually require longer parity ladders, so the aggregate resource comparison depends on the Hamiltonian and the synthesis method.

### Constraint-preserving mixer

Start with exactly one 1 in each one-hot row. An XY mixer exchanges an excitation between two policy qubits in the same row and preserves that row's number of 1s. With a feasible initial state and ideal gates, this keeps probability in the feasible subspace. It can trade penalty terms for more involved state preparation or mixer gates.

## Takeaway

Fewer qubits do not imply an easier circuit. Compare the feasible fraction, the number and order of interactions, and the gate counts under a stated synthesis method. The table describes logical synthesis estimates for these Hamiltonians.

## Presentation asset

Contrast the 4×2 and 8×8 row pairs. Show qubits, highest-order interactions, and the estimated CNOT count beside the common policy-map objective.

MkDocs topic for this notebook: `docs/benchmarking.md`.

## Next step

Phase 9, `09_hardware_aware_transpilation.ipynb`, will compare these logical estimates with circuit resources after mapping to representative device constraints.

### References

[1] F. Glover, G. Kochenberger and Y. Du, [A Tutorial on Formulating and Using QUBO Models](https://arxiv.org/abs/1811.11538), 2018 preprint.